In [1]:
import sys
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, when, regexp_replace, lit, coalesce
from pyspark.sql.types import LongType, IntegerType, StringType
from delta import *

# ===================================================
# 1. CONFIGURAÇÃO (SPARK + DELTA)
# ===================================================
warehouse_location = "hdfs://hdfs-nn:9000/warehouse"

print("🚀 A iniciar Spark Session para BoxOffice...")

builder = SparkSession.builder \
    .appName("BoxOffice ETL (Bronze -> Silver)") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print("✅ Sessão iniciada.")

# ===================================================
# 2. UPLOAD DO FICHEIRO (Segurança)
# ===================================================
try:
    from hdfs import InsecureClient
    client = InsecureClient("http://hdfs-nn:9870", user="anonymous")
    
    local_csv = "boxoffice.csv" 
    bronze_path = "/demo/bronze/boxoffice.csv"
    
    if os.path.exists(local_csv):
        print(f"⬆️ A enviar {local_csv} para o HDFS...")
        client.upload(bronze_path, local_csv, overwrite=True)
        print("✅ Upload concluído.")
    else:
        print(f"⚠️ Ficheiro local '{local_csv}' não encontrado. A assumir que já existe no HDFS.")
except ImportError:
    print("⚠️ Biblioteca 'hdfs' não instalada. A saltar verificação de upload.")

# ===================================================
# 3. LEITURA E LIMPEZA
# ===================================================
hdfs_path = "hdfs://hdfs-nn:9000/demo/bronze/boxoffice.csv"
print(f"\n📖 A ler Bronze: {hdfs_path}")

try:
    df_bronze = spark.read \
        .option("header", "true") \
        .option("delimiter", ",") \
        .csv(hdfs_path)
except Exception as e:
    print("❌ Erro ao ler CSV. Verifica se o ficheiro 'boxoffice.csv' existe no Bronze.")
    raise e

print("🧹 A transformar dados...")

# Aplicar a tua lógica de limpeza (Gross -> revenue)
# E padronizar nomes de colunas (Title -> title)
df_silver = df_bronze \
    .withColumn("title", trim(col("Title"))) \
    .withColumn("year", col("Year").cast(IntegerType())) \
    .withColumn(
        "revenue", # Renomeei de 'gross_clean' para 'revenue' para ser mais claro
        when(
            col("Gross").isNotNull() & (col("Gross") != ""),
            regexp_replace(
                col("Gross").cast(StringType()),
                r"[^0-9]", # Remove tudo o que não for número ($ e ,)
                ""
            ).cast(LongType())
        ).otherwise(lit(None).cast(LongType()))
    ) \
    .select("title", "year", "revenue") # Selecionar apenas as colunas finais limpas

# Filtro básico: remover se não tiver Título ou Ano
df_silver = df_silver.filter(col("title").isNotNull() & col("year").isNotNull())

print("✅ Dados transformados.")

# ===================================================
# 4. GRAVAÇÃO (DELTA)
# ===================================================
# Caminho alinhado: /warehouse/silver.db/boxoffice
silver_path = f"{warehouse_location}/silver.db/boxoffice"

print(f"\n💾 A gravar Tabela Delta em: {silver_path}")

# Apagar tabela antiga
spark.sql("DROP TABLE IF EXISTS silver.boxoffice")

df_silver.write \
    .mode("overwrite") \
    .format("delta") \
    .partitionBy("year") \
    .option("overwriteSchema", "true") \
    .save(silver_path)

# ===================================================
# 5. REGISTO NO HIVE
# ===================================================
# Criar base de dados se não existir
spark.sql(f"CREATE DATABASE IF NOT EXISTS silver LOCATION '{warehouse_location}/silver.db'")

spark.sql(f"""
    CREATE TABLE silver.boxoffice
    USING DELTA
    LOCATION '{silver_path}'
""")

print("🎉 Tabela 'silver.boxoffice' criada com sucesso!")

# ===================================================
# 6. VERIFICAÇÃO FINAL
# ===================================================
print("\n=== Top 5 Filmes com maior Bilheteira ===")
spark.sql("""
    SELECT year, title, format_number(revenue, 0) as receita 
    FROM silver.boxoffice 
    WHERE revenue IS NOT NULL 
    ORDER BY revenue DESC 
    LIMIT 5
""").show(truncate=False)

🚀 A iniciar Spark Session para BoxOffice...
✅ Sessão iniciada.
⚠️ Ficheiro local 'boxoffice.csv' não encontrado. A assumir que já existe no HDFS.

📖 A ler Bronze: hdfs://hdfs-nn:9000/demo/bronze/boxoffice.csv
🧹 A transformar dados...
✅ Dados transformados.

💾 A gravar Tabela Delta em: hdfs://hdfs-nn:9000/warehouse/silver.db/boxoffice
🎉 Tabela 'silver.boxoffice' criada com sucesso!

=== Top 5 Filmes com maior Bilheteira ===
+----+------------------------------------------+-------------+
|year|title                                     |receita      |
+----+------------------------------------------+-------------+
|2019|Avengers: Endgame                         |2,799,439,100|
|2009|Avatar                                    |2,743,577,587|
|2022|Avatar: The Way of Water                  |2,320,250,281|
|2015|Star Wars: Episode VII - The Force Awakens|2,068,223,624|
|2018|Avengers: Infinity War                    |2,048,359,754|
+----+------------------------------------------+------------